# Comparison of all setups in one convenient table (unthrottled)

In [1]:
import pandas as pd;
import numpy as np;
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
import scipy.stats as stats

plt.rcParams.update({'font.size': 14})
pd.set_option('display.float_format', lambda x: '%.2f' % x)
path = '../../../../playwright/results/core-web-vitals/testrun-9/'


In [2]:
features = ['navTime', 'totalTime', 'lcp', 'fcp', 'ttfb', 'tbt', 'tti', 'longestTask', 'longTasks', 'nf:init', 'nf:config','nf:loaded']

dirty_dfs =	{
  "Monolith": pd.read_csv(f'{path}results-monolith.csv', sep=',').iloc[5:],
  "CSR": pd.read_csv(f'{path}results-csr.csv', sep=',').iloc[5:],
  "CSR sd": pd.read_csv(f'{path}results-csr-sd.csv', sep=',').iloc[5:],
  "SSRH": pd.read_csv(f'{path}results-ssrh.csv', sep=',').iloc[5:],
  "SSRH sd": pd.read_csv(f'{path}results-ssrh-sd.csv', sep=',').iloc[5:],
  "SSRV": pd.read_csv(f'{path}results-ssrv-sd.csv', sep=',').iloc[5:],
}

In [3]:
def detect_outliers(_df, _features, contamination=0.1):
    clf = IsolationForest(contamination=contamination, random_state=42)
    outliers = clf.fit_predict(_df[_features])
    return outliers == 1

masks = {}
dfs = {}
target_features = ['navTime', 'totalTime', 'lcp', 'fcp', 'ttfb']

for name, _df in dirty_dfs.items():
    mask = detect_outliers(_df, target_features)
    masks[name] = mask
    dfs[name] = _df[mask].copy()

In [4]:
columns = [ 'ttfb','fcp','nf:init','lcp','tti','nf:loaded','tbt','longestTask']
rows = []

for name, df in dfs.items():
    mean_row = df[columns].mean()
    rows.append((f"{name} (mean)", mean_row))

for name, df in dfs.items():
    percentile_row = df[columns].quantile(0.75)
    rows.append((f"{name} (75th)", percentile_row))

result_df = pd.DataFrame([row[1] for row in rows], index=[row[0] for row in rows])
result_df = result_df.mask(result_df < 0, '-')
result_df

,ttfb,fcp,nf:init,lcp,tti,nf:loaded,tbt,longestTask
Monolith (mean),3.56,73.52,-,73.52,73.52,-,0.00,-
CSR (mean),3.28,68.92,43.10,141.55,68.92,109.85,0.00,-
CSR sd (mean),3.13,68.45,42.67,156.67,68.45,125.01,0.00,-
SSRH (mean),16.55,92.36,59.58,92.36,92.36,120.99,0.00,-
SSRH sd (mean),17.22,92.14,61.19,92.14,92.14,141.44,0.00,-
SSRV (mean),28.51,82.27,-,91.85,82.27,-,0.00,-
Monolith (75th),3.80,75.48,-,75.48,75.48,-,0.00,-
CSR (75th),3.50,70.67,44.48,144.27,70.67,111.60,0.00,-
CSR sd (75th),3.40,70.70,43.80,160.38,70.70,127.20,0.00,-
SSRH (75th),17.70,95.70,61.80,95.70,95.70,124.30,0.00,-
